In [1]:
# ============================================================
# STRATEGY 3 OPTION B — 256×256
# Exports: best_model.pth, model_fp32.onnx, model_int8.onnx
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, random, os, json, time
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────
# SNN Components
# ──────────────────────────────────────────────
class SurrogateGradient(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return input.gt(0).float()
    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        temp = 10.0
        return grad_output / (temp * torch.abs(input) + 1.0) ** 2

spike_fn = SurrogateGradient.apply

class LIFNeuron(nn.Module):
    def __init__(self, tau=3.0, threshold=0.5, soft_reset=True):
        super().__init__()
        self.tau = tau
        self.threshold = threshold
        self.soft_reset = soft_reset
        self.membrane = None

    def forward(self, x):
        if self.membrane is None:
            self.membrane = torch.zeros_like(x)
        new_membrane = self.membrane * (1 - 1 / self.tau) + x
        spikes = spike_fn(new_membrane - self.threshold)
        if self.soft_reset:
            self.membrane = new_membrane - spikes * self.threshold
        else:
            self.membrane = new_membrane * (1 - spikes)
        return spikes

    def reset(self):
        self.membrane = None

# ──────────────────────────────────────────────
# Building Blocks
# ──────────────────────────────────────────────
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, stride=1, pad=1):
        super().__init__()
        if in_ch < 4:
            self.conv = nn.Conv2d(in_ch, out_ch, k, stride=stride, padding=pad, bias=False)
            self.bn = nn.BatchNorm2d(out_ch)
            self.dw = False
        else:
            self.depthwise  = nn.Conv2d(in_ch, in_ch, k, stride=stride, padding=pad, groups=in_ch, bias=False)
            self.pointwise  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
            self.bn         = nn.BatchNorm2d(out_ch)
            self.dw         = True

    def forward(self, x):
        x = self.depthwise(x) if self.dw else self.conv(x)
        if self.dw: x = self.pointwise(x)
        return self.bn(x)

class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        rd = max(1, ch // r)
        self.fc = nn.Sequential(nn.Linear(ch, rd, bias=False), nn.ReLU(True),
                                 nn.Linear(rd, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        a = self.fc(self.avg(x).view(b, c))
        m = self.fc(self.max(x).view(b, c))
        return x * (a + m).view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k//2, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, 1, keepdim=True)
        mx, _ = torch.max(x, 1, keepdim=True)
        return x * self.sig(self.conv(torch.cat([avg, mx], 1)))

class EfficientResidualBlock(nn.Module):
    def __init__(self, ch, spiking=True):
        super().__init__()
        self.c1  = DepthwiseSeparableConv(ch, ch)
        self.a1  = LIFNeuron() if spiking else nn.ReLU(False)
        self.c2  = DepthwiseSeparableConv(ch, ch)
        self.a2  = LIFNeuron() if spiking else nn.ReLU(False)
        self.att = ChannelAttention(ch, 8)
        self.spiking = spiking

    def forward(self, x):
        out = self.a1(self.c1(x))
        out = self.a2(self.c2(out) + x)
        return self.att(out)

    def reset(self):
        if self.spiking:
            for a in [self.a1, self.a2]:
                if hasattr(a, 'reset'): a.reset()

class CompactDownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, spiking=True):
        super().__init__()
        self.down = DepthwiseSeparableConv(in_ch, out_ch, stride=2)
        self.act  = LIFNeuron() if spiking else nn.ReLU(False)
        self.r1   = EfficientResidualBlock(out_ch, spiking)
        self.r2   = EfficientResidualBlock(out_ch, spiking)
        self.sa   = SpatialAttention()
        self.spiking = spiking

    def forward(self, x):
        return self.sa(self.r2(self.r1(self.act(self.down(x)))))

    def reset(self):
        if self.spiking and hasattr(self.act, 'reset'): self.act.reset()
        self.r1.reset(); self.r2.reset()

class CompactUpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, skip_ch, spiking=True):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch//2, 4, stride=2, padding=1, bias=False)
        self.bn1  = nn.BatchNorm2d(in_ch//2)
        self.au   = LIFNeuron() if spiking else nn.ReLU(False)
        self.comb = nn.Conv2d(in_ch//2 + skip_ch, out_ch, 3, padding=1, bias=False)
        self.bn2  = nn.BatchNorm2d(out_ch)
        self.ac   = LIFNeuron() if spiking else nn.ReLU(False)
        self.r1   = EfficientResidualBlock(out_ch, spiking)
        self.r2   = EfficientResidualBlock(out_ch, spiking)
        self.spiking = spiking

    def forward(self, x, skip):
        x = self.au(self.bn1(self.up(x)))
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = self.ac(self.bn2(self.comb(torch.cat([x, skip], 1))))
        return self.r2(self.r1(x))

    def reset(self):
        if self.spiking:
            for a in [self.au, self.ac]:
                if hasattr(a, 'reset'): a.reset()
            self.r1.reset(); self.r2.reset()

class DownsampledSkipConnection(nn.Module):
    def __init__(self, ch, target=(8,8), comp_ch=8):
        super().__init__()
        self.target = target
        self.compress   = nn.Conv2d(ch, comp_ch, 1, bias=False)
        self.bn_c       = nn.BatchNorm2d(comp_ch)
        self.decompress = nn.Conv2d(comp_ch, ch, 1, bias=False)
        self.bn_d       = nn.BatchNorm2d(ch)
        self.gate = nn.Sequential(
            nn.Conv2d(ch*2, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(False),
            nn.Conv2d(ch, ch, 1, bias=False),  nn.BatchNorm2d(ch), nn.Sigmoid()
        )
        self.gate_scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        orig = x.shape[2:]
        xs   = F.adaptive_avg_pool2d(x, self.target) if orig != self.target else x
        comp = self.bn_c(self.compress(xs))
        dec  = self.bn_d(self.decompress(comp))
        if orig != self.target:
            dec = F.interpolate(dec, size=orig, mode='bilinear', align_corners=False)
        gate    = self.gate(torch.cat([dec, x], 1))
        refined = dec + torch.tanh(self.gate_scale) * gate * (x - dec)
        return comp, refined

# ──────────────────────────────────────────────
# Encoder — 256×256 channel map:
# init   : 3→32  (256×256)
# down1  : 32→48 (128×128)
# down2  : 48→64 ( 64×64)
# down3  : 64→48 ( 32×32)
# down4  : 48→32 ( 16×16)
# down5  : 32→24 (  8×8 )
# bottleneck: 24→16 (8×8)
# ──────────────────────────────────────────────
class Strategy3Encoder(nn.Module):
    def __init__(self, in_ch=3, time_steps=2):
        super().__init__()
        self.T   = time_steps
        self.ic  = nn.Conv2d(in_ch, 32, 3, padding=1, bias=False)
        self.ibn = nn.BatchNorm2d(32)
        self.ia  = nn.ReLU(False)
        self.d1  = CompactDownBlock(32, 48,  spiking=False)
        self.d2  = CompactDownBlock(48, 64,  spiking=True)
        self.d3  = CompactDownBlock(64, 48,  spiking=True)
        self.d4  = CompactDownBlock(48, 32,  spiking=True)
        self.d5  = CompactDownBlock(32, 24,  spiking=True)
        self.bn  = nn.Sequential(
            DepthwiseSeparableConv(24, 16), nn.ReLU(False),
            EfficientResidualBlock(16, True), EfficientResidualBlock(16, True)
        )
        self.sk1 = DownsampledSkipConnection(32)
        self.sk2 = DownsampledSkipConnection(48)
        self.sk3 = DownsampledSkipConnection(64)
        self.sk4 = DownsampledSkipConnection(48)
        self.sk5 = DownsampledSkipConnection(32)

    def forward(self, x):
        x   = self.ia(self.ibn(self.ic(x)))
        enc = sd = sc = None
        for _ in range(self.T):
            s1=x;       x1=self.d1(s1)
            s2=x1;      x2=self.d2(s2)
            s3=x2;      x3=self.d3(s3)
            s4=x3;      x4=self.d4(s4)
            s5=x4;      x5=self.d5(s5)
            b = self.bn(x5)
            c1,d1=self.sk1(s1); c2,d2=self.sk2(s2); c3,d3=self.sk3(s3)
            c4,d4=self.sk4(s4); c5,d5=self.sk5(s5)
            if enc is None:
                enc=[b]; sc=[[c1,c2,c3,c4,c5]]; sd=[[d1,d2,d3,d4,d5]]
            else:
                enc.append(b)
                sc.append([c1,c2,c3,c4,c5]); sd.append([d1,d2,d3,d4,d5])
        T=self.T
        enc_out = sum(enc)/T
        sd_out  = [sum(sd[t][i] for t in range(T))/T for i in range(5)]
        sc_out  = [sum(sc[t][i] for t in range(T))/T for i in range(5)]
        return enc_out, sd_out, sc_out

    def reset(self):
        for blk in [self.d1,self.d2,self.d3,self.d4,self.d5]: blk.reset()
        for m in self.bn:
            if hasattr(m,'reset'): m.reset()

# ──────────────────────────────────────────────
# Decoder
# up1: 16→8  + skip5(32) → 24
# up2: 24→12 + skip4(48) → 32
# up3: 32→16 + skip3(64) → 48
# up4: 48→24 + skip2(48) → 64
# up5: 64→32 + skip1(32) → 48
# ──────────────────────────────────────────────
class Strategy3Decoder(nn.Module):
    def __init__(self, out_ch=3, time_steps=2):
        super().__init__()
        self.T  = time_steps
        self.u1 = CompactUpBlock(16, 24, 32, spiking=True)
        self.u2 = CompactUpBlock(24, 32, 48, spiking=True)
        self.u3 = CompactUpBlock(32, 48, 64, spiking=True)
        self.u4 = CompactUpBlock(48, 64, 48, spiking=True)
        self.u5 = CompactUpBlock(64, 48, 32, spiking=False)
        self.rf = nn.Sequential(
            DepthwiseSeparableConv(48, 32), nn.ReLU(False),
            nn.Conv2d(32, out_ch, 1, bias=True)
        )

    def forward(self, enc, skips):
        recon = None
        for _ in range(self.T):
            x = self.u1(enc, skips[4])
            x = self.u2(x,   skips[3])
            x = self.u3(x,   skips[2])
            x = self.u4(x,   skips[1])
            x = self.u5(x,   skips[0])
            x = self.rf(x)
            recon = x if recon is None else recon + x
        return torch.sigmoid(recon / self.T)

    def reset(self):
        for blk in [self.u1,self.u2,self.u3,self.u4,self.u5]: blk.reset()

class Strategy3Autoencoder(nn.Module):
    def __init__(self, in_ch=3, time_steps=2):
        super().__init__()
        self.encoder = Strategy3Encoder(in_ch, time_steps)
        self.decoder = Strategy3Decoder(in_ch, time_steps)

    def forward(self, x):
        enc, sd, sc = self.encoder(x)
        return self.decoder(enc, sd), enc, sc

    def reset(self):
        self.encoder.reset(); self.decoder.reset()

# ONNX wrapper — returns only reconstruction
class ONNXWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        dec, _, _ = self.model(x)
        return dec

# ──────────────────────────────────────────────
# Loss
# ──────────────────────────────────────────────
class EnhancedPerceptualLoss(nn.Module):
    def __init__(self, alpha=0.84):
        super().__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        self.l1  = nn.L1Loss()

    def ssim_loss(self, x, y):
        mx=F.avg_pool2d(x,3,1,1); my=F.avg_pool2d(y,3,1,1)
        sx=F.avg_pool2d(x**2,3,1,1)-mx**2; sy=F.avg_pool2d(y**2,3,1,1)-my**2
        sxy=F.avg_pool2d(x*y,3,1,1)-mx*my
        C1,C2=0.01**2,0.03**2
        return 1-(((2*mx*my+C1)*(2*sxy+C2))/((mx**2+my**2+C1)*(sx+sy+C2)+1e-8)).mean()

    def forward(self, out, tgt):
        grad_loss = (self.l1(out[:,:,:,1:]-out[:,:,:,:-1], tgt[:,:,:,1:]-tgt[:,:,:,:-1]) +
                     self.l1(out[:,:,1:,:]-out[:,:,:-1,:], tgt[:,:,1:,:]-tgt[:,:,:-1,:]))
        return ((1-self.alpha)*self.mse(out,tgt) + self.alpha*self.ssim_loss(out,tgt) +
                0.1*(torch.mean((out.mean([2,3])-tgt.mean([2,3]))**2)+
                     torch.mean((out.std([2,3]) -tgt.std([2,3]))**2)) +
                0.05*grad_loss)

# ──────────────────────────────────────────────
# Dataset
# ──────────────────────────────────────────────
class VideoFrameDataset(Dataset):
    def __init__(self, paths, size=256, n=50, aug=True):
        self.paths=paths; self.size=size; self.n=n; self.aug=aug
    def __len__(self): return len(self.paths)*self.n
    def __getitem__(self, idx):
        cap=cv2.VideoCapture(str(self.paths[idx//self.n]))
        total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total>0: cap.set(cv2.CAP_PROP_POS_FRAMES, (idx%self.n)%max(total,1))
        ret,frame=cap.read(); cap.release()
        if not ret or frame is None:
            return torch.zeros(3,self.size,self.size)
        frame=cv2.resize(frame,(self.size,self.size))
        frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB).astype(np.float32)/255.
        if self.aug:
            if np.random.rand()>.5: frame=np.fliplr(frame).copy()
            if np.random.rand()>.5: frame=np.clip(frame*np.random.uniform(.85,1.15),0,1)
        return torch.from_numpy(frame.transpose(2,0,1))

# ──────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────
def collect_videos(root, classes, max_per=30):
    root=Path(root); paths=[]
    for c in classes:
        d=root/c
        if not d.exists(): continue
        vids=list(d.glob('**/*.avi'))+list(d.glob('**/*.mp4'))
        paths.extend(random.sample(vids,min(max_per,len(vids))))
    return paths

def calc_psnr(o,r):
    return float(np.mean([psnr(a.cpu().numpy().transpose(1,2,0),
                               b.cpu().numpy().transpose(1,2,0),data_range=1.)
                           for a,b in zip(o,r)]))

def calc_ssim(o,r):
    return float(np.mean([ssim(a.cpu().numpy().transpose(1,2,0),
                               b.cpu().numpy().transpose(1,2,0),data_range=1.,channel_axis=2)
                           for a,b in zip(o,r)]))

def t2rgb(t):
    return (np.clip(t.cpu().numpy().transpose(1,2,0),0,1)*255).astype(np.uint8)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def model_mb(m):
    p=sum(p.nelement()*p.element_size() for p in m.parameters())
    b=sum(b.nelement()*b.element_size() for b in m.buffers())
    return (p+b)/(1024**2)

# ──────────────────────────────────────────────
# Export Functions
# ──────────────────────────────────────────────
def export_pth(model, path):
    torch.save(model.state_dict(), path)
    size = os.path.getsize(path)/(1024**2)
    print(f"✅ PTH saved → {path}  ({size:.2f} MB)")

def export_fp32_onnx(model, path, res=256):
    model.eval(); model.reset()
    wrapper = ONNXWrapper(model).cpu().eval()
    dummy = torch.randn(1, 3, res, res)
    torch.onnx.export(
        wrapper, dummy, path,
        export_params=True, opset_version=17,
        do_constant_folding=True,
        input_names=['input'], output_names=['reconstruction'],
        dynamic_axes={'input':{0:'batch'}, 'reconstruction':{0:'batch'}},
        dynamo=False
    )
    # Verify
    import onnx
    onnx.checker.check_model(onnx.load(path))
    size = os.path.getsize(path)/(1024**2)
    print(f"✅ FP32 ONNX saved → {path}  ({size:.2f} MB)")

def export_int8_onnx(fp32_path, int8_path, res=256, n_calib=100):
    """
    Dynamic INT8 quantization — no calibration data needed.
    For better accuracy, static quantization with real frames is also shown.
    """
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(
        model_input=fp32_path,
        model_output=int8_path,
        weight_type=QuantType.QInt8,
        optimize_model=True
    )
    size = os.path.getsize(int8_path)/(1024**2)
    print(f"✅ INT8 ONNX saved → {int8_path}  ({size:.2f} MB)")

def benchmark_onnx(path, res=256, n=50, label=''):
    import onnxruntime as ort
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = 1
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess = ort.InferenceSession(path, sess_options=opts,
                                providers=['CPUExecutionProvider'])
    dummy = np.random.randn(1,3,res,res).astype(np.float32)
    iname = sess.get_inputs()[0].name
    for _ in range(10): sess.run(None,{iname:dummy})
    times = []
    for _ in range(n):
        t0=time.perf_counter(); sess.run(None,{iname:dummy})
        times.append((time.perf_counter()-t0)*1000)
    print(f"  {label:15s} | {np.mean(times):.1f}ms avg | {1000/np.mean(times):.1f} FPS")
    return np.mean(times)

# ──────────────────────────────────────────────
# Training
# ──────────────────────────────────────────────
def train(model, tr_loader, val_loader, epochs, device, exp_dir, patience=7, res=256):
    criterion = EnhancedPerceptualLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    best_loss=float('inf'); p_cnt=0
    history={'tr':[],'val':[],'psnr':[],'ssim':[],'epoch':[]}

    for ep in range(epochs):
        model.train(); tr_loss=0
        for frames in tqdm(tr_loader, desc=f'Ep{ep+1}/{epochs} Train'):
            frames=frames.to(device); model.reset()
            recon,_,_=model(frames); loss=criterion(recon,frames)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); tr_loss+=loss.item()

        model.eval(); val_loss=0; ps=[]; ss=[]
        with torch.no_grad():
            for frames in tqdm(val_loader, desc=f'Ep{ep+1}/{epochs} Val  '):
                frames=frames.to(device); model.reset()
                r,_,_=model(frames); val_loss+=criterion(r,frames).item()
                ps.append(calc_psnr(frames,r)); ss.append(calc_ssim(frames,r))

        atr=tr_loss/len(tr_loader); av=val_loss/len(val_loader)
        ap=float(np.mean(ps)); as_=float(np.mean(ss))
        history['tr'].append(atr); history['val'].append(av)
        history['psnr'].append(ap); history['ssim'].append(as_)
        history['epoch'].append(ep+1)

        improved=(best_loss-av)>1e-4
        print(f"Ep{ep+1:3d}: tr={atr:.4f} val={av:.4f} PSNR={ap:.2f}dB SSIM={as_:.4f} "
              f"{'✓ BEST' if improved else f'({p_cnt+1}/{patience})'}")
        if improved:
            best_loss=av; p_cnt=0
            torch.save(model.state_dict(), os.path.join(exp_dir,'best_model.pth'))
        else:
            p_cnt+=1
            if p_cnt>=patience:
                print(f"⏹ Early stop ep{ep+1}"); break
        sched.step()

    fig,axes=plt.subplots(1,3,figsize=(15,4))
    axes[0].plot(history['epoch'],history['tr'],label='Train')
    axes[0].plot(history['epoch'],history['val'],label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(history['epoch'],history['psnr']); axes[1].set_title('PSNR')
    axes[2].plot(history['epoch'],history['ssim']); axes[2].set_title('SSIM')
    plt.tight_layout()
    plt.savefig(os.path.join(exp_dir,'curves.png'),dpi=100); plt.close()
    return history

# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────
def main():
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    RES = 256
    EXP = f'strategy3_256x256'
    os.makedirs(EXP, exist_ok=True)

    UCF_ROOT = Path("/kaggle/input/datasets/pevogam/ucf101/UCF101/UCF-101")
    CLASSES  = ["Basketball","Biking","Diving","MilitaryParade","SkateBoarding",
                "SoccerJuggling","TennisSwing","WalkingWithDog","HorseRiding","Drumming"]

    videos = collect_videos(UCF_ROOT, CLASSES, max_per=30)
    random.shuffle(videos)
    n=len(videos); n_tr=int(.7*n); n_val=int(.15*n)
    tr_v=videos[:n_tr]; val_v=videos[n_tr:n_tr+n_val]; te_v=videos[n_tr+n_val:]
    print(f"Split → train:{len(tr_v)} val:{len(val_v)} test:{len(te_v)}")

    tr_ds  = VideoFrameDataset(tr_v,  RES, 50, aug=True)
    val_ds = VideoFrameDataset(val_v, RES, 50, aug=False)
    te_ds  = VideoFrameDataset(te_v,  RES, 50, aug=False)

    tr_l  = DataLoader(tr_ds,  8, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
    val_l = DataLoader(val_ds, 8, shuffle=False, num_workers=2, pin_memory=True)
    te_l  = DataLoader(te_ds,  4, shuffle=False, num_workers=2, pin_memory=True)

    model = Strategy3Autoencoder(time_steps=2).to(device)
    print(f"Params: {count_params(model)/1e6:.3f}M  Size: {model_mb(model):.2f}MB")

    # ── Train ──────────────────────────────────────────────────────
    train(model, tr_l, val_l, epochs=30, device=device, exp_dir=EXP, res=RES)

    # ── Load best and test ─────────────────────────────────────────
    model.load_state_dict(torch.load(f'{EXP}/best_model.pth', map_location=device, weights_only=True))
    model.eval()
    ps=[]; ss=[]; lats=[]; origs=[]; recons=[]
    with torch.no_grad():
        for frames in tqdm(te_l, desc='Testing'):
            frames=frames.to(device); model.reset()
            if device.type=='cuda': torch.cuda.synchronize()
            t0=time.time(); r,_,_=model(frames)
            if device.type=='cuda': torch.cuda.synchronize()
            lats.append((time.time()-t0)*1000/frames.size(0))
            ps.append(calc_psnr(frames,r)); ss.append(calc_ssim(frames,r))
            for i in range(frames.size(0)):
                if len(origs)<100:
                    origs.append(t2rgb(frames[i])); recons.append(t2rgb(r[i]))

    print(f"\n{'='*55}")
    print(f" TEST RESULTS — Strategy 3 Option B (256×256)")
    print(f"{'='*55}")
    print(f" Parameters : {count_params(model)/1e6:.3f}M")
    print(f" Model size : {model_mb(model):.2f} MB")
    print(f" PSNR       : {np.mean(ps):.2f} dB")
    print(f" SSIM       : {np.mean(ss):.4f}")
    print(f" Latency    : {np.mean(lats):.2f} ms/frame  (GPU)")
    print(f" Throughput : {1000/np.mean(lats):.1f} FPS  (GPU)")
    print(f"{'='*55}")

    # ── Save PNG comparisons ───────────────────────────────────────
    os.makedirs(f'{EXP}/frames', exist_ok=True)
    for i in range(min(17,len(origs))):
        fig,ax=plt.subplots(1,2,figsize=(10,5))
        ax[0].imshow(origs[i]);   ax[0].set_title(f'Original {i+1}');   ax[0].axis('off')
        ax[1].imshow(recons[i]);  ax[1].set_title(f'Reconstructed {i+1}'); ax[1].axis('off')
        plt.tight_layout()
        plt.savefig(f'{EXP}/frames/frame_{i+1:03d}.png',dpi=100,bbox_inches='tight')
        plt.close()

    # ── Export PTH ────────────────────────────────────────────────
    export_pth(model, f'{EXP}/best_model.pth')

    # ── Export FP32 ONNX ──────────────────────────────────────────
    fp32_path = f'{EXP}/model_fp32.onnx'
    export_fp32_onnx(model, fp32_path, res=RES)

    # ── Export INT8 ONNX ──────────────────────────────────────────
    int8_path = f'{EXP}/model_int8.onnx'
    export_int8_onnx(fp32_path, int8_path, res=RES)

    # ── Benchmark both ONNX on CPU (Kaggle) ───────────────────────
    print(f"\n{'='*55}")
    print(f" ONNX CPU BENCHMARK (Kaggle, single-thread)")
    print(f"{'='*55}")
    benchmark_onnx(fp32_path, res=RES, label='FP32 ONNX')
    benchmark_onnx(int8_path, res=RES, label='INT8 ONNX')
    print(f"{'='*55}")
    print(f"\n📦 Files to copy to Raspberry Pi 4:")
    print(f"   {fp32_path}")
    print(f"   {int8_path}")
    print(f"   (optional) {EXP}/best_model.pth")

if __name__ == '__main__':
    main()

Split → train:210 val:45 test:45
Params: 0.536M  Size: 2.07MB


Ep1/30 Val  : 100%|██████████| 282/282 [01:32<00:00,  3.06it/s]


Ep  1: tr=0.0854 val=0.0371 PSNR=26.34dB SSIM=0.9605 ✓ BEST


Ep2/30 Val  : 100%|██████████| 282/282 [01:32<00:00,  3.05it/s]


Ep  2: tr=0.0287 val=0.0313 PSNR=32.01dB SSIM=0.9710 ✓ BEST


...

Ep27/30 Val  : 100%|██████████| 282/282 [01:31<00:00,  3.07it/s]


Ep 27: tr=0.0066 val=0.0109 PSNR=41.45dB SSIM=0.9932 (7/7)
⏹ Early stop ep27


Testing: 100%|██████████| 563/563 [01:41<00:00,  5.52it/s]



 TEST RESULTS — Strategy 3 Option B (256×256)
 Parameters : 0.536M
 Model size : 2.07 MB
 PSNR       : 40.08 dB
 SSIM       : 0.9986
 Latency    : 31.91 ms/frame  (GPU)
 Throughput : 31.3 FPS  (GPU)
✅ PTH saved → strategy3_256x256/best_model.pth  (2.27 MB)
✅ FP32 ONNX saved → strategy3_256x256/model_fp32.onnx  (2.57 MB)


ModuleNotFoundError: No module named 'onnxruntime'

In [2]:
# Cell 1 — Install
import subprocess
subprocess.run(['pip', 'install', 'onnxruntime', 'onnx', '-q'], check=True)
print("✅ Installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 65.6 MB/s eta 0:00:00
✅ Installed


In [5]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process
import os

FP32_PATH      = '/kaggle/working/strategy3_256x256/model_fp32.onnx'
FP32_PREP_PATH = 'strategy3_256x256/model_fp32_prep.onnx'   # preprocessed intermediate
INT8_PATH      = 'strategy3_256x256/model_int8.onnx'

# Step 1 — Preprocess (shape inference + graph optimization)
print("Step 1: Preprocessing FP32 ONNX...")
quant_pre_process(
    input_model_path  = FP32_PATH,
    output_model_path = FP32_PREP_PATH,
    skip_optimization = False,
    skip_onnx_shape   = False,
    skip_symbolic_shape = False,
    auto_merge_symbolic_dims = True,
    verbose = 0
)
print(f"✅ Preprocessed model saved → {FP32_PREP_PATH}")

# Step 2 — Quantize the preprocessed model
print("\nStep 2: Quantizing to INT8...")
quantize_dynamic(
    model_input  = FP32_PREP_PATH,
    model_output = INT8_PATH,
    weight_type  = QuantType.QInt8
)

# Sizes
fp32_mb = os.path.getsize(FP32_PATH)      / (1024**2)
prep_mb = os.path.getsize(FP32_PREP_PATH) / (1024**2)
int8_mb = os.path.getsize(INT8_PATH)      / (1024**2)

print(f"\n✅ INT8 ONNX saved → {INT8_PATH}")
print(f"   FP32 original   : {fp32_mb:.2f} MB")
print(f"   FP32 preprocessed: {prep_mb:.2f} MB")
print(f"   INT8 quantized  : {int8_mb:.2f} MB")
print(f"   Size reduction  : {fp32_mb/int8_mb:.2f}× smaller")


Step 1: Preprocessing FP32 ONNX...
✅ Preprocessed model saved → strategy3_256x256/model_fp32_prep.onnx

Step 2: Quantizing to INT8...

✅ INT8 ONNX saved → strategy3_256x256/model_int8.onnx
   FP32 original   : 2.57 MB
   FP32 preprocessed: 2.64 MB
   INT8 quantized  : 1.84 MB
   Size reduction  : 1.39× smaller


In [6]:
# Cell 3 — Verify + Benchmark both models on CPU
import onnxruntime as ort
import numpy as np
import time

def benchmark(path, res=256, n=50, label=''):
    opts = ort.SessionOptions()
    opts.intra_op_num_threads     = 1
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess  = ort.InferenceSession(path, sess_options=opts,
                                 providers=['CPUExecutionProvider'])
    dummy = np.random.randn(1, 3, res, res).astype(np.float32)
    iname = sess.get_inputs()[0].name
    for _ in range(10): sess.run(None, {iname: dummy})   # warmup
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        sess.run(None, {iname: dummy})
        times.append((time.perf_counter() - t0) * 1000)
    avg = np.mean(times)
    print(f"  {label:18s} | {avg:7.1f} ms avg | {1000/avg:6.1f} FPS  (CPU)")

RES = 256
print(f"{'='*55}")
print(f"  CPU BENCHMARK (single-thread, {RES}×{RES})")
print(f"{'='*55}")
benchmark(FP32_PATH, res=RES, label='FP32 ONNX')
benchmark(INT8_PATH, res=RES, label='INT8 ONNX')
print(f"{'='*55}")
print(f"\n📦 Copy both to Raspberry Pi 4:")
print(f"   {FP32_PATH}")
print(f"   {INT8_PATH}")


  CPU BENCHMARK (single-thread, 256×256)
  FP32 ONNX          |   451.1 ms avg |    2.2 FPS  (CPU)
  INT8 ONNX          |  3243.2 ms avg |    0.3 FPS  (CPU)

📦 Copy both to Raspberry Pi 4:
   /kaggle/working/strategy3_256x256/model_fp32.onnx
   strategy3_256x256/model_int8.onnx
